# Probability Density Functions — Interactive Notebook

Concept **24-pdf**. Stdlib-only exploration of:

1. The Gaussian PDF $\phi(x) = \tfrac{1}{\sqrt{2\pi}} e^{-x^2/2}$ and its numerical normalisation.
2. The exponential PDF and inverse-CDF sampling.
3. Why a PDF can exceed 1 (density vs. probability).


## 1. Setup

We use only `math` and `random` from the standard library, no NumPy or SciPy.


In [ ]:
import math, random

def gauss_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

def exp_pdf(x, lam):
    return lam * math.exp(-lam * x) if x >= 0 else 0.0

def trapz(f, a, b, n):
    h = (b - a) / n
    s = 0.5 * (f(a) + f(b))
    for k in range(1, n):
        s += f(a + k * h)
    return h * s

print('phi(0)  =', gauss_pdf(0.0))
print('phi(1)  =', gauss_pdf(1.0))


## 2. Gaussian normalisation via the trapezoidal rule

We integrate $\phi$ over $[-5, 5]$. The tails outside this interval carry less than $6\times 10^{-7}$ of mass.


In [ ]:
integral = trapz(gauss_pdf, -5.0, 5.0, 10_000)
print('trapz(phi, -5, 5) =', integral)
print('|1 - integral|   =', abs(1.0 - integral))


## 3. Inverse-CDF sampling for the exponential distribution

For $X \sim \mathrm{Exp}(\lambda)$, the CDF is $F(x) = 1 - e^{-\lambda x}$ and the inverse is $F^{-1}(u) = -\tfrac{1}{\lambda}\log(1 - u)$. Feed in $U \sim \mathrm{Uniform}(0,1)$ and out come exponential samples.


In [ ]:
random.seed(0)
lam = 1.5
N = 100_000
samples = [-math.log(1.0 - random.random()) / lam for _ in range(N)]

bin_w = 0.25
n_bins = 20
counts = [0] * n_bins
for s in samples:
    if 0.0 <= s < n_bins * bin_w:
        counts[int(s / bin_w)] += 1

print(f'{"center":>7}  {"empirical":>10}  {"pdf":>10}  {"|diff|":>8}')
for k in range(n_bins):
    c = (k + 0.5) * bin_w
    emp = counts[k] / (N * bin_w)
    pdf = exp_pdf(c, lam)
    print(f'{c:7.3f}  {emp:10.5f}  {pdf:10.5f}  {abs(emp-pdf):8.4f}')


## 4. Densities can exceed 1

A PDF is *probability per unit length*, not probability. Uniform on $[0, 0.01]$ has density $100$.


In [ ]:
def narrow_uniform_pdf(x, a=0.0, b=0.01):
    return 1.0 / (b - a) if a <= x <= b else 0.0

print('density at x=0.005:', narrow_uniform_pdf(0.005))
print('integral over support =',
      trapz(narrow_uniform_pdf, -0.5, 0.5, 100_000))


## 5. Sanity check: empirical mean of Exp(λ)

Theory says $\mathbb{E}[X] = 1/\lambda$. We compare.


In [ ]:
emp_mean = sum(samples) / len(samples)
print('empirical mean =', emp_mean)
print('theoretical 1/lambda =', 1.0 / lam)
print('relative error =', abs(emp_mean - 1.0/lam) / (1.0/lam))


## 6. Takeaways

- A PDF is a non-negative integrable function with $\int f = 1$.
- $f(x)\,dx$ is infinitesimal probability mass near $x$ — the value $f(x)$ is **not** itself a probability.
- Inverse-CDF sampling turns any $U \sim \mathrm{Uniform}(0,1)$ into a sample from any continuous distribution whose CDF you can invert.
- Numerical integration of the Gaussian PDF recovers the constraint $\int \phi = 1$ to machine precision over $[-5, 5]$.
